In [1]:
"""
过敏状态转移模型演示
"""

import sys
import os
from pathlib import Path

# 获取当前 notebook 所在的目录（Jupyter 中）
try:
    # 尝试获取 __file__（在 .py 文件中有效）
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 在 Jupyter notebook 中，使用当前工作目录
    current_dir = os.getcwd()
    
# 将当前目录添加到系统路径
sys.path.append(current_dir)

from models.antagonistic_model import AntagonisticStateTransitionModel
from models.base_model import EventType

In [2]:
def demo_antagonistic_model():
    """演示拮抗状态转移模型"""
    print("=" * 80)
    print("拮抗状态转移模型演示")
    print("=" * 80)
    
    # 创建模型实例 - n=3 (需要3次A和3次B交替)
    model = AntagonisticStateTransitionModel(n=3)
    
    n_val = model.model_params['n']
    
    print(f"\n📊 模型信息:")
    print(f"   参数: n={n_val}")
    print(f"   初始状态: (1,1,1) 不健康")
    print(f"   完全转移条件: 交替应用干预A和干预B，每种各{n_val}次（共{2*n_val}步）")
    print(f"   可用干预:")
    print(f"      - 干预A: (0,0,1) 🔵")
    print(f"      - 干预B: (0,1,0) 🟢")
    print(f"      - 同时应用: (0,1,1) 🟣 (拮抗，状态不变)")
    print(f"      - 无干预: (0,0,0) ⚪")
    
    print(f"\n📖 状态说明:")
    print(f"   (1,1,1) → 不健康状态")
    print(f"   (0,0,0) → 稳定健康状态")
    
    print(f"\n📖 事件说明:")
    print(f"   E7a (完全转移): 存在长度为{2*n_val}的交替序列 → 健康")
    print(f"   E7b (无有效转移): 其他情况 → 不健康")
    
    print(f"\n📖 交替模式:")
    print(f"   模式1: A, B, A, B, ... (以A开始)")
    print(f"   模式2: B, A, B, A, ... (以B开始)")
    
    # 场景1: 正确交替序列 (模式1)
    print("\n" + "=" * 80)
    print(f"场景1: 正确交替序列 (模式1: A,B,A,B,..., 共{2*n_val}步)")
    print("=" * 80)
    
    interventions_scene1 = []
    for i in range(n_val):
        interventions_scene1.append((0, 0, 1))  # A
        interventions_scene1.append((0, 1, 0))  # B
    
    result1 = model.simulate(interventions_scene1, return_details=True)
    _print_results(result1, model, title=f"正确交替{2*n_val}步 → 健康")
    
    # 场景2: 正确交替序列 (模式2)
    print("\n" + "=" * 80)
    print(f"场景2: 正确交替序列 (模式2: B,A,B,A,..., 共{2*n_val}步)")
    print("=" * 80)
    
    interventions_scene2 = []
    for i in range(n_val):
        interventions_scene2.append((0, 1, 0))  # B
        interventions_scene2.append((0, 0, 1))  # A
    
    result2 = model.simulate(interventions_scene2, return_details=True)
    _print_results(result2, model, title=f"正确交替{2*n_val}步 → 健康")
    
    # 场景3: 不完整的交替序列
    print("\n" + "=" * 80)
    print(f"场景3: 不完整的交替序列 ({2*n_val-2}步)")
    print("=" * 80)
    
    interventions_scene3 = []
    for i in range(n_val - 1):
        interventions_scene3.append((0, 0, 1))  # A
        interventions_scene3.append((0, 1, 0))  # B
    
    result3 = model.simulate(interventions_scene3, return_details=True)
    _print_results(result3, model, title=f"不完整交替{2*n_val-2}步 → 不健康")
    
    # 场景4: 包含拮抗干预
    print("\n" + "=" * 80)
    print("场景4: 包含拮抗干预 (同时应用A和B)")
    print("=" * 80)
    
    interventions_scene4 = [
        (0, 0, 1),  # A
        (0, 1, 0),  # B
        (0, 1, 1),  # AB (拮抗)
        (0, 0, 1),  # A
        (0, 1, 0),  # B
        (0, 0, 1),  # A
        (0, 1, 0),  # B
    ]
    
    result4 = model.simulate(interventions_scene4, return_details=True)
    _print_results(result4, model, title="包含拮抗干预 → 破坏交替序列，不健康")
    
    # 场景5: 序列中间有中断
    print("\n" + "=" * 80)
    print("场景5: 交替序列中间有中断")
    print("=" * 80)
    
    interventions_scene5 = [
        (0, 0, 1),  # A
        (0, 1, 0),  # B
        (0, 0, 1),  # A
        (0, 0, 0),  # 无干预 (中断)
        (0, 1, 0),  # B
        (0, 0, 1),  # A
        (0, 1, 0),  # B
    ]
    
    result5 = model.simulate(interventions_scene5, return_details=True)
    _print_results(result5, model, title="交替序列中断 → 不健康")


def demo_alternating_progress():
    """演示交替序列进度跟踪"""
    print("\n" + "=" * 80)
    print("交替序列进度跟踪演示")
    print("=" * 80)
    
    model = AntagonisticStateTransitionModel(n=4)
    n_val = model.model_params['n']
    
    print(f"\n目标: 需要 {2*n_val} 步正确交替 (n={n_val})")
    print("-" * 60)
    
    # 逐步构建交替序列
    test_sequences = []
    
    # 序列1: 部分交替
    seq1 = [(0, 0, 1), (0, 1, 0), (0, 0, 1)]
    test_sequences.append(("部分交替 (3步)", seq1))
    
    # 序列2: 接近完成
    seq2 = []
    for i in range(n_val - 1):
        seq2.append((0, 0, 1))
        seq2.append((0, 1, 0))
    test_sequences.append((f"接近完成 ({2*n_val-2}步)", seq2))
    
    # 序列3: 完整交替
    seq3 = []
    for i in range(n_val):
        seq3.append((0, 0, 1))
        seq3.append((0, 1, 0))
    test_sequences.append((f"完整交替 ({2*n_val}步)", seq3))
    
    for name, seq in test_sequences:
        progress = model.check_alternating_progress(seq)
        print(f"\n{name}:")
        print(f"   最长交替长度: {progress['longest_alternating_length']}")
        print(f"   模式类型: {progress['pattern_type']}")
        print(f"   还需步数: {progress['needed_for_complete']}")
        print(f"   是否完成: {'是' if progress['is_complete'] else '否'}")


def demo_antagonistic_effect():
    """演示拮抗效应"""
    print("\n" + "=" * 80)
    print("拮抗效应演示")
    print("=" * 80)
    
    model = AntagonisticStateTransitionModel(n=3)
    
    # 测试同时应用干预的情况
    print("\n测试: 同时应用干预A和干预B (拮抗)")
    interventions = [
        (0, 1, 1),  # 同时应用
        (0, 1, 1),  # 同时应用
        (0, 1, 1),  # 同时应用
    ]
    
    result = model.simulate(interventions, return_details=True)
    
    print(f"\n干预序列:")
    for t, inter in enumerate(interventions, 1):
        print(f"   t={t}: {inter} 🟣 同时应用")
    
    print(f"\n状态演变:")
    print("-" * 60)
    print(f"{'步数':<6} {'可观测状态':<20} {'事件':<30}")
    print("-" * 60)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        state_name = "健康" if state == model.HEALTHY else "不健康"
        
        if i == 0:
            event_str = "-"
        else:
            event = result["events"][i-1]
            if event == EventType.COMPLETE_TRANSITION:
                event_str = "E7a (完全转移)"
            else:
                event_str = "E7b (无有效转移)"
        
        display_state = f"{state} ({state_name})"
        print(f"{i:<6} {display_state:<20} {event_str:<30}")
    
    print(f"\n💡 观察:")
    print(f"   - 同时应用干预A和干预B时，状态保持不变")
    print(f"   - 这是因为拮抗作用阻止了状态变化")


def _print_results(result: dict, model, title: str = ""):
    """打印模拟结果"""
    if title:
        print(f"\n{title}")
    
    print(f"\n干预序列:")
    for t, inter in enumerate(result["interventions"], 1):
        if inter == model.INTERVENTION_A:
            print(f"   t={t:2d}: {inter} 🔵 干预A")
        elif inter == model.INTERVENTION_B:
            print(f"   t={t:2d}: {inter} 🟢 干预B")
        elif inter == model.INTERVENTION_AB:
            print(f"   t={t:2d}: {inter} 🟣 同时应用(拮抗)")
        else:
            print(f"   t={t:2d}: {inter} ⚪ 无干预")
    
    print(f"\n状态演变:")
    print("-" * 80)
    print(f"{'步数':<6} {'可观测状态':<20} {'状态说明':<15} {'事件':<40}")
    print("-" * 80)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        state_name = "健康" if state == model.HEALTHY else "不健康"
        
        if i == 0:
            event_str = "-"
        else:
            event = result["events"][i-1]
            if event == EventType.COMPLETE_TRANSITION:
                event_str = "E7a (完全转移) ⭐"
            else:
                event_str = "E7b (无有效转移)"
        
        display_state = f"{state} ({state_name})"
        print(f"{i:<6} {display_state:<20} {state_name:<15} {event_str:<40}")
    
    # 显示交替序列进度
    progress = model.check_alternating_progress(result["interventions"])
    if progress["longest_alternating_length"] > 0:
        print(f"\n📊 交替序列进度:")
        print(f"   最长交替长度: {progress['longest_alternating_length']}/{2*model.model_params['n']}")
        print(f"   模式类型: {progress['pattern_type']}")
        if not progress["is_complete"] and progress["needed_for_complete"] > 0:
            print(f"   还需 {progress['needed_for_complete']} 步完成交替")

In [3]:
if __name__ == "__main__":
    demo_antagonistic_model()
    demo_alternating_progress()
    demo_antagonistic_effect()

拮抗状态转移模型演示

📊 模型信息:
   参数: n=3
   初始状态: (1,1,1) 不健康
   完全转移条件: 交替应用干预A和干预B，每种各3次（共6步）
   可用干预:
      - 干预A: (0,0,1) 🔵
      - 干预B: (0,1,0) 🟢
      - 同时应用: (0,1,1) 🟣 (拮抗，状态不变)
      - 无干预: (0,0,0) ⚪

📖 状态说明:
   (1,1,1) → 不健康状态
   (0,0,0) → 稳定健康状态

📖 事件说明:
   E7a (完全转移): 存在长度为6的交替序列 → 健康
   E7b (无有效转移): 其他情况 → 不健康

📖 交替模式:
   模式1: A, B, A, B, ... (以A开始)
   模式2: B, A, B, A, ... (以B开始)

场景1: 正确交替序列 (模式1: A,B,A,B,..., 共6步)

正确交替6步 → 健康

干预序列:
   t= 1: (0, 0, 1) 🔵 干预A
   t= 2: (0, 1, 0) 🟢 干预B
   t= 3: (0, 0, 1) 🔵 干预A
   t= 4: (0, 1, 0) 🟢 干预B
   t= 5: (0, 0, 1) 🔵 干预A
   t= 6: (0, 1, 0) 🟢 干预B

状态演变:
--------------------------------------------------------------------------------
步数     可观测状态                状态说明            事件                                      
--------------------------------------------------------------------------------
0      (1, 1, 1) (不健康)      不健康             -                                       
1      (1, 1, 1) (不健康)      不健康             E7b (无有效转移)              